In [25]:
import requests
import pandas as pd
from sklearn.model_selection import train_test_split

In [26]:
class RussianOllamaClient:
    def __init__(self, model_name="llama3:latest"):
        self.model_name = model_name
        self.base_url = "http://localhost:11434"
        print(f"Используем модель: {model_name}")
    
    def make_request(self, prompt, temperature):
        data = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": temperature,
                "num_predict": 100
            }
        }
        
        try:
            response = requests.post(
                f"{self.base_url}/api/generate", 
                json=data, 
                timeout=500
            )
            
            if response.status_code == 200:
                result = response.json()
                return result['response']
            else:
                print(f"❌ Ошибка {response.status_code}")
                return ""
                
        except Exception as e:
            print(f"❌ Исключение: {e}")
            return ""

In [28]:
client = RussianOllamaClient("llama3:8b")  
    
test_response = client.make_request("Скажи 'тест пройден'", temperature=0.3)
if test_response:
    print("✅ Модель работает")
else:
    print("❌ Модель не отвечает")

Используем модель: llama3:8b
✅ Модель работает


In [29]:
try:
    df = pd.read_csv('ru_cefr_short.csv')
    print("✅ Файл найден:")
except:
    print("❌ Файл не найден, используем тестовые данные:")
    test_data = {
        'fragment': [
            "Весной, летом и осенью почти каждую субботу он играет в футбол. У них в школе есть футбольная команда. А зимой он играет в хоккей. Ещё мы любим театр.",
            "Вчера я весь вечер повторял грамматику, учил слова. В контрольной работе я сделал 3 ошибки. Питер и Кен написали всё без ошибок. Преподаватель сказал, что они делают большие успехи.",
            "В такой ситуации особенно сложно работающим студентам, которые должны находить время и для учёбы, и для работы. Это нередко вызывает стресс.",
            "Как редкое животное они стоили очень дорого и являлись предметом роскоши. Археологи нашли останки этих животных в развалинах домов богатых людей четвёртого века нашей эры.",
            "Он увлёкся энтомологией ещё мальчиком и в детстве познакомился с основными учёными трудами в этой области.",
            "Так же радиация — это частица, которая летит на огромной скорости. Сама частица может быть почти любой: от атомного ядра до электрона или фотона."
        ],
        'textbook-assigned cefr level': [1, 2, 3, 4, 5, 6]
    }
    df = pd.DataFrame(test_data)

df

✅ Файл найден:


,fragment,textbook-assigned cefr level
0,"Весной, летом и осенью почти каждую субботу он...",1
1,"Все говорят, что мама хорошая хозяйка. А ещё н...",1
2,На каждой двери красные плакаты и красные фона...,1
3,"Я считаю деньги, в час обедаю в кафе, а потом ...",1
4,Магазин «Чёрный квадрат» открывается в 9 часов...,1
...,...,...
7317,Утечка мозгов стала ключевым трендом междунаро...,6
7318,"По оценкам менеджеров «Промы», такая ситуация ...",6
7319,"Но это не мы, а техно-мемы заполоняют мир благ...",6
7320,Mapillary использует программное обеспечение д...,6


In [30]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['fragment'].values,
    df['textbook-assigned cefr level'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['textbook-assigned cefr level']
)

len(train_texts)

5857

In [31]:
texts_to_augment = []

for i in range(len(train_labels)):
  if train_labels[i] == 6:
    texts_to_augment.append(train_texts[i])


len(texts_to_augment)

120

# 11. Перефразирование

In [32]:
def create_prompt(text):
    return f"""Перефразируй этот текст на русском языке, сохраняя его уровень сложности. Текст должен оставаться на том же уровне сложности.

Пример:

Оригинал: 'Искусственный интеллект меняет способы взаимодействия людей с технологиями.'

Аугментация: 'Технологии искусственного интеллекта трансформируют методы коммуникации человека с цифровыми устройствами.'

Текст: '{text}'

Верни ТОЛЬКО новый текст на русском языке без пояснений и комментариев, не заключай новый текст в кавычки:"""

In [33]:
def augment_11(temperature):
    df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

    n = 1
    print(f"Аугментируем {len(texts_to_augment)} текстов...")
    
    for text in texts_to_augment:
    
        prompt = create_prompt(text)
        augmented_text = client.make_request(prompt, temperature)
        
        print(f"\n{n}. Реальный текст:\n\t{text}")
        print(f"Аугметированный текст:\n\t{augmented_text}")
       
        new_row = pd.DataFrame({
            'text': [text],
            'augmented-text': [augmented_text]
        })
        df_pred = pd.concat([df_pred, new_row], ignore_index=True)
    
        n+=1
    
    df_pred.to_csv(f"c2_augmented_11_llama3_temp_{str(temperature).replace('.', '_')}.csv")

## Temperature = 0.0

In [34]:
temperature = 0.0
augment_11(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	4:38–4:54 В природе сама радиация не заразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также используются для лечения людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о вложении миллиарда долларов в собственную систему автоматизированного управления транспортными средствами, а компания Waymo уже запустила приложение для владельцев их самоуправляемых машин на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки и

## Temperature = 0.1

In [35]:
temperature = 0.1
augment_11(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	4:38–4:54 В природе сама радиация не заразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также используются для лечения людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о вложении миллиарда долларов в собственную систему автоматизированного управления транспортными средствами, а компания Waymo уже запустила приложение для владельцев самоуправляемых машин на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из п

## Temperature = 0.2

In [36]:
temperature = 0.2
augment_11(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	4:38–4:54 В природе самой по себе радиация не заразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также используются для лечения людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о вложении одного миллиарда долларов в собственную систему автоматизированного управления транспортными средствами, а компания Waymo уже запустила приложение для владельцев их самоуправляемых автомобилей на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дяд

## Temperature = 0.3

In [37]:
temperature = 0.3
augment_11(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	11:38–11:54 В самом себе радиация не заразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о вложении миллиарда долларов в собственную систему автоматизированного управления транспортными средствами, а компания Waymo уже запустила приложение для владельцев их самоуправляемых машин на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из подполья» и други

## Temperature = 0.4

In [38]:
temperature = 0.4
augment_11(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	4:38–4:54 В природе радиация сама по себе не заражает. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также используются для лечения людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о вложении в миллиард долларов в собственную систему автоматизированного управления транспортными средствами, а компания Waymo запустила приложение для владельцев самоуправляемых машин на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записк

## Temperature = 0.5

In [39]:
temperature = 0.5
augment_11(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	Сама по себе радиация не заражает. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о вложении миллиарда долларов в собственную систему автоматизированного управления транспортными средствами, а компания Waymo уже запустила приложение для владельцев их самоуправляемых автомобилей на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из подполья» и другие, ра

## Temperature = 0.6

In [40]:
temperature = 0.6
augment_11(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	7:38–7:54 Сама по себе радиация является безопасной. В частности, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также используются для лечения людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о значительной инвестиции в миллиард долларов в собственное решение автоматизированного управления транспортными средствами, а компания Waymo уже запустила приложение для владельцев своих самоуправляемых машин на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник

## Temperature = 0.7

In [41]:
temperature = 0.7
augment_11(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	Самой по себе радиация несвязана с заражением. Например, значительные дозы излучения от тех же изотопов часто стерилизуют фрукты и овощи, а также лечат людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила об инвестировании в миллиард долларов в собственное решение автоматизации вождения, а компания Waymo уже запустила приложение для владельцев своих самоуправляемых автомобилей на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из подполья» и другие, рассказы, публи

## Temperature = 0.8

In [42]:
temperature = 0.8
augment_11(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	7:38-7:54 Вtrinsic радиация не заразна. Например, мощные дозы излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также используются для лечения людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о масштабной инвестиции в миллиард долларов в развитие собственной системы автоматизированного управления транспортными средствами, а компания Waymo первой среди игроков на рынке самоуправляемых машин запустила приложение для владельцев таких автомобилей на платформе Android.

3. Реальный текст:
	Мн

## Temperature = 0.9

In [43]:
temperature = 0.9
augment_11(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	5:38–5:54 В изначальном виде радиация сама по себе не заражает. К примеру, значительными дозами облучения от тех же самых изотопов фрукты и овощи мощной дозой стерилизуют, а людей лечат.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила об инвестировании в свою систему автоматизированного управления вождения сумму в один миллиард долларов, а компания Waymo запустила приложение для владельцев самоуправляемых машин на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из по

## Temperature = 1.0

In [44]:
temperature = 1.0
augment_11(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	9:28-9:44 Сама радиация в природе иноконтактна. К примеру, мощные дозы излучения от тех же изотопов часто стерилизуют фрукты и овощи, а также используются для лечения людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о вложении миллиарда долларов в свою систему автоматизированного управления движением, а компания Waymo уже запустила приложение для владельцев их самоуправляющихся машин на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из подполья» и другие, расс

## Temperature = 1.1

In [45]:
temperature = 1.1
augment_11(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	5:38–5:54 Сами по себе лучи радиации инокулены. Например, высокими дозами излучения от тех же самых изотопов обычно стерилизуют фрукты и овощи, а также используются для лечения людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о внесении инвестиций в размере одного миллиарда долларов в систему автоматизированного управления, а компания Waymo запустила приложение для Android-устройств владельцев своих самоуправляемых машин.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из подполь

## Temperature = 1.2

In [46]:
temperature = 1.2
augment_11(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	10:46–11:02 Сама по себе радиация нетвяжущая. Например, мощными дозами излучения от тех же самых изотопов частично стерилизуют фрукты и овощи, а также используются в медицинских процедурах для лечения пациентов.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о вложении один миллиард долларов в свою систему автоматизированного управления автомобилями, а компания Waymo запустила приложение для владельцев самоуправляемых транспортных средств на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «

## Temperature = 1.3

In [47]:
temperature = 1.3
augment_11(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	4:27-4:43 Сама по себе радиация не передается. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют сельскохозяйственную продукцию и используются для лечения людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber недавно заявила о вложении одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo уже запустила приложение для владельцев их самоуправляемых машин на платформе Android.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из п

## Temperature = 1.4

In [48]:
temperature = 1.4
augment_11(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	5:20–5:36 В природе сам по себе излучение радиацией не заразно. Например, значительными дозами луча от тех же самых изотопов фрукты и овощи регулярно стерилизуют, а людей лечат.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber объявила о масштабной инвестиции в свою систему автоматизированного управления транспортными средствами, достигающую одного миллиарда долларов. Стоит отметить, что компания Waymo уже запустила на Android приложение для владельцев своих самоуправляемых машин.

3. Реальный текст:
	Множество повестей: «Д

## Temperature = 1.5

In [49]:
temperature = 1.5
augment_11(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
Аугметированный текст:
	11:28–11:44 Сама по себе радиация бестерильна. К примеру, значительными дозами излучения от тех же самых изотопов фрукты и овощи мощной дозой обрабатывают, а также лечат людей.

2. Реальный текст:
	Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Аугметированный текст:
	Компания Uber заявила об инвестировании миллиарда долларов в свой алгоритм автопилота, а компания Waymo запустила приложение для Android, предназначенное для владельцев ее самоуправляющихся машин.

3. Реальный текст:
	Множество повестей: «Двойник», «Дядюшкин сон», «Записки из подполья» и другие, рассказы, публицисти